# Project: Australian Market Expansion Strategy - 
## Workforce-Based Business Expansion Analytics

### Description:

Goal: explore and clean the datasets
Tech: Python\
Library: pandas, numpy, re, string\
Data Source: Australian Bureau of Statistics\
Datasets: EmploymentByState_NewSouthWales, EmploymentByState_NorthernTerritory\
Import Funtions Files: Functions.py

## Part1. Library and datasets

### Library:

In [1]:
# install
#!pip install pandas openpyxl

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import pandas as pd
import numpy as np
import os
import re
import string
from pathlib import Path
from Functions import extract_metadata, melt_and_update_columns, merge_metadata_and_melted_dataset, update_column_names

### Files:

Location-

In [4]:
fileLocation = '/Users/abbottting/Desktop/Project/AMES/OriginalData' 

In [61]:
#CPIMonthly_path = fileLocation + '/' + "CPI_MonthlyPercentageChange.xlsx"
#EmployPopulationRatial_Path = fileLocation + '/' + 'EmploymentToPopulationRatio.csv'
filepath = {
    'EmploymentByState_NewSouthWales_Path' : {
        'file': fileLocation + '/'  + 'EmploymentByState_NewSouthWales.xlsx', 
        'sheet': 'Data1'
    },
    'EmploymentByState_NorthernTerritory_Path' : {
        'file': fileLocation + '/'  + 'EmploymentByState_NorthernTerritory.xlsx', 
        'sheet': 'Data1'
    },
    'EmploymentByState_Queensland_Path' : {
        'file': fileLocation + '/'  + 'EmploymentByState_Queensland.xlsx',
        'sheet': 'Data1'
    },
    'EmploymentByState_SouthAustralia_Path' : {
        'file': fileLocation + '/' + 'EmploymentByState_SouthAustralia.xlsx',
        'sheet': 'Data1'
    },
    'EmploymentByState_Tasmania_Path' : {
        'file': fileLocation + '/' + 'EmploymentByState_Tasmania.xlsx',
        'sheet': 'Data1'
    },
    'EmploymentByState_Victoria_Path' : {
        'file': fileLocation + '/' + 'EmploymentByState_Victoria.xlsx',
        'sheet': 'Data1'
    },
    'EmploymentByState_WesternAuds_Path' : {
        'file': fileLocation + '/' + 'EmploymentByState_WesternAuds.xlsx',
        'sheet': 'Data1'
    },
    'ERP_Quarterly_Path' : {
        'file':fileLocation + '/' + 'ERP_Quarterly_unfiltered.csv'
    },
    'Job_vacancies_Path' : {
        'file': fileLocation + '/' + 'Job_vacancies.xlsx',
        'sheet': 'Sheet1'
    }
}

#Create a status tracker where every file starts as False
status_tracker = {key: False for key in filepath}

print(f"Tracking initialized. Total files to read: {len(status_tracker)}")

Tracking initialized. Total files to read: 9


Read files-

In [63]:
dataframes = {}

for key, path in filepath.items():
    #Quick check to ensure the file exists physically before reading
    file_path = Path(path['file'])
    if os.path.exists(file_path):
        clean_name = key.replace('_Path', '_Original')

        if file_path.suffix.lower() == ".csv" :
            dataframes[clean_name] = pd.read_csv(file_path)
            status_tracker[key] = True  # Flip status to True ONLY if read succeeds
            
        elif file_path.suffix.lower() == '.xlsx':
            sheet_name = path['sheet']
            try:
                dataframes[clean_name] = pd.read_excel(file_path, sheet_name)
            except:
                print(f"Can't detect the sheet: {sheet_name}")
            else:
                status_tracker[key] = True  # Flip status to True ONLY if read succeeds
        else:
            print(f"Can't detect the file type of this file: {file_path}")
            
    else:
        print(f"!!!Warning: File not found at: {file_path}")


#Count how many files were successfully read
successful_reads = sum(status_tracker.values())
expected_reads = len(filepath)

print(f"Expected files to read: {expected_reads}")
print(f"Successfully read files: {successful_reads}")

# Final Verification Check
if successful_reads == expected_reads:
    print("Success: Every single file path was successfully read!")
else:
    print("Error: Some file paths were skipped or failed to read.")
    # Show exactly which files failed
    for key, status in status_tracker.items():
        if not status:
            print(f"   - Failed to read: {key}")


Expected files to read: 9
Successfully read files: 9
Success: Every single file path was successfully read!


In [64]:
for key in dataframes:
    print(key)

EmploymentByState_NewSouthWales_Original
EmploymentByState_NorthernTerritory_Original
EmploymentByState_Queensland_Original
EmploymentByState_SouthAustralia_Original
EmploymentByState_Tasmania_Original
EmploymentByState_Victoria_Original
EmploymentByState_WesternAuds_Original
ERP_Quarterly_Original
Job_vacancies_Original


## Part2. Data Extraction and transformation

### A. EmploymentByState_NewSouthWales

#### Observe and Analyse the dataset

In [46]:
dataframes['EmploymentByState_NewSouthWales_Original'].head(10)

,Unnamed: 0,Employed total ; Persons ;,Employed total ; Persons ;.1,Employed total ; Persons ;.2,Employed total ; > Males ;,Employed total ; > Males ;.1,Employed total ; > Males ;.2,Employed total ; > Females ;,Employed total ; > Females ;.1,Employed total ; > Females ;.2,...,Participation rate ; > Males ;.2,Participation rate ; > Females ;,Participation rate ; > Females ;.1,Participation rate ; > Females ;.2,Not in the labour force (NILF) ; Persons ;,Not in the labour force (NILF) ; > Males ;,Not in the labour force (NILF) ; > Females ;,Civilian population aged 15 years and over ; Persons ;,Civilian population aged 15 years and over ; > Males ;,Civilian population aged 15 years and over ; > Females ;
0,Unit,000,000,000,000,000,000,000,000,000,...,Percent,Percent,Percent,Percent,000,000,000,000,000,000
1,Series Type,Trend,Seasonally Adjusted,Original,Trend,Seasonally Adjusted,Original,Trend,Seasonally Adjusted,Original,...,Original,Trend,Seasonally Adjusted,Original,Original,Original,Original,Original,Original,Original
2,Data Type,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,...,PERCENT,PERCENT,PERCENT,PERCENT,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK
3,Frequency,Month,Month,Month,Month,Month,Month,Month,Month,Month,...,Month,Month,Month,Month,Month,Month,Month,Month,Month,Month
4,Collection Month,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
5,Series Start,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,...,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00
6,Series End,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,...,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00
7,No. Obs,576,576,576,576,576,576,576,576,576,...,576,576,576,576,576,576,576,576,576,576
8,Series ID,A84423937R,A84423265K,A84423601K,A84423825W,A84423153T,A84423489W,A84424049K,A84423377C,A84423713C,...,A84423495T,A84424055F,A84423383X,A84423719T,A84423604T,A84423492K,A84423716K,A84423605V,A84423493L,A84423717L
9,1978-02-01 00:00:00,2108.211696,2113.937155,2114.116622,1363.006258,1365.43433,1369.400359,745.205437,748.502825,744.716262,...,79.757877,42.768707,43.022597,43.388591,1447.67271,371.382833,1076.289877,3735.891949,1834.702955,1901.188995


In [47]:
dataframes['EmploymentByState_NewSouthWales_Original'].columns

Index(['Unnamed: 0', 'Employed total ;  Persons ;',
       'Employed total ;  Persons ;.1', 'Employed total ;  Persons ;.2',
       'Employed total ;  > Males ;', 'Employed total ;  > Males ;.1',
       'Employed total ;  > Males ;.2', 'Employed total ;  > Females ;',
       'Employed total ;  > Females ;.1', 'Employed total ;  > Females ;.2',
       '> Employed full-time ;  Persons ;',
       '> Employed full-time ;  Persons ;.1',
       '> Employed full-time ;  Persons ;.2',
       '> Employed full-time ;  > Males ;',
       '> Employed full-time ;  > Males ;.1',
       '> Employed full-time ;  > Males ;.2',
       '> Employed full-time ;  > Females ;',
       '> Employed full-time ;  > Females ;.1',
       '> Employed full-time ;  > Females ;.2',
       '> Employed part-time ;  Persons ;',
       '> Employed part-time ;  > Males ;',
       '> Employed part-time ;  > Females ;',
       'Employment to population ratio ;  Persons ;',
       'Employment to population ratio ;  Persons ;.

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Step1. Turn the first to eighth row index into mata data

In [233]:
# Extract the first eight rows into metadata and transpose the matrix
mata_data = dataframes['EmploymentByState_NewSouthWales_Original'].iloc[:9].copy().T
mata_data.head()

,0,1,2,3,4,5,6,7,8
Unnamed: 0,Unit,Series Type,Data Type,Frequency,Collection Month,Series Start,Series End,No. Obs,Series ID
Employed total ; Persons ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423937R
Employed total ; Persons ;.1,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423265K
Employed total ; Persons ;.2,000,Original,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423601K
Employed total ; > Males ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423825W


In [234]:
# reset index
mata_data = mata_data.reset_index()
mata_data.head()

,index,0,1,2,3,4,5,6,7,8
0,Unnamed: 0,Unit,Series Type,Data Type,Frequency,Collection Month,Series Start,Series End,No. Obs,Series ID
1,Employed total ; Persons ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423937R
2,Employed total ; Persons ;.1,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423265K
3,Employed total ; Persons ;.2,000,Original,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423601K
4,Employed total ; > Males ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423825W


In [235]:
# extract first row and use it as column names
new_columns = list(mata_data.iloc[0])
new_columns

['Unnamed: 0',
 'Unit',
 'Series Type',
 'Data Type',
 'Frequency',
 'Collection Month',
 'Series Start',
 'Series End',
 'No. Obs',
 'Series ID']

In [90]:
# update column format
for i in range(len(new_columns)):
    column_name = new_columns[i].lower()
    punctuation_removed = column_name.translate(str.maketrans('','',string.punctuation))
    new_columns[i] = punctuation_removed.replace(' ', '_')

In [91]:
# assign, remove first row, reset index
mata_data.columns = new_columns
mata_data_updated_columns = mata_data.iloc[1:].reset_index(drop = True)
mata_data_updated_columns.head(5)

,unnamed_0,unit,series_type,data_type,frequency,collection_month,series_start,series_end,no_obs,series_id
0,Employed total ; Persons ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423937R
1,Employed total ; Persons ;.1,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423265K
2,Employed total ; Persons ;.2,000,Original,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423601K
3,Employed total ; > Males ;,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423825W
4,Employed total ; > Males ;.1,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423153T


In [92]:
# update the first column name
first_column = mata_data_updated_columns.columns[0]
print(f'original column name: {first_column}')
mata_data_updated_columns.rename(columns={first_column : 'variable'}, inplace = True)
print(f'new column name: {mata_data_updated_columns.columns[0]}')
# assign values to a list in first column
variables_original = list(mata_data_updated_columns['variable'])

original column name: unnamed_0
new column name: variable


In [93]:
# Split the variable with genders
variables = []
genders = []
for i, o in enumerate(variables_original):
    splited_object = o.split(';')
    #get the variable and gender from the string
    one_variable = splited_object[0].strip()
    strip_chars = string.whitespace + string.punctuation + string.digits
    one_gender = splited_object[1].strip(strip_chars)
    variables.append(one_variable)
    genders.append(one_gender)

# add gender and variable to the dataset
mata_data_updated_columns.insert(1, 'gender', genders)
mata_data_updated_columns['variable'] = variables

In [94]:
mata_data_updated_columns.head()

,variable,gender,unit,series_type,data_type,frequency,collection_month,series_start,series_end,no_obs,series_id
0,Employed total,Persons,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423937R
1,Employed total,Persons,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423265K
2,Employed total,Persons,000,Original,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423601K
3,Employed total,Males,000,Trend,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423825W
4,Employed total,Males,000,Seasonally Adjusted,STOCK,Month,1,1978-02-01 00:00:00,2026-01-01 00:00:00,576,A84423153T


#### Step2. use series ID as column, prepare to melt the rest of dataset including series id, date, and values

In [95]:
# check if series_id has no duplicate
duplicate_rows = mata_data_updated_columns[mata_data_updated_columns.duplicated(subset=['series_id'], keep=False)]
duplicate_rows.shape

(0, 11)

In [96]:
value = EmploymentByState_NewSouthWales_Original.iloc[9:]

In [97]:
#check data
value.head().iloc[:, :5]

,Unnamed: 0,Employed total ; Persons ;,Employed total ; Persons ;.1,Employed total ; Persons ;.2,Employed total ; > Males ;
9,1978-02-01 00:00:00,2108.211696,2113.937155,2114.116622,1363.006258
10,1978-03-01 00:00:00,2109.021527,2094.809924,2110.179431,1361.723258
11,1978-04-01 00:00:00,2109.854462,2107.530544,2115.063279,1360.306263
12,1978-05-01 00:00:00,2111.157961,2111.430838,2111.268984,1358.992234
13,1978-06-01 00:00:00,2112.979788,2112.439636,2113.247138,1357.54763


In [98]:
# Get series ID
series_id = EmploymentByState_NewSouthWales_Original.loc[8]

In [99]:
# use series ID as new columnnames
value_series_id = value.rename(columns = series_id)
value_series_id.head().iloc[:, :5]

,Series ID,A84423937R,A84423265K,A84423601K,A84423825W
9,1978-02-01 00:00:00,2108.211696,2113.937155,2114.116622,1363.006258
10,1978-03-01 00:00:00,2109.021527,2094.809924,2110.179431,1361.723258
11,1978-04-01 00:00:00,2109.854462,2107.530544,2115.063279,1360.306263
12,1978-05-01 00:00:00,2111.157961,2111.430838,2111.268984,1358.992234
13,1978-06-01 00:00:00,2112.979788,2112.439636,2113.247138,1357.54763


In [100]:
# update the first column name to 'date'
value_date = value_series_id.rename(columns = {'Series ID': 'date'})

In [101]:
#melt the data set
value_melted = pd.melt(value_date, id_vars = 'date', var_name = 'series_id', value_name='value')
value_melted.head()

,date,series_id,value
0,1978-02-01,A84423937R,2108.211696
1,1978-03-01,A84423937R,2109.021527
2,1978-04-01,A84423937R,2109.854462
3,1978-05-01,A84423937R,2111.157961
4,1978-06-01,A84423937R,2112.979788


In [102]:
value_melted['state'] = 'NewSouthWales'
value_melted.head()

,date,series_id,value,state
0,1978-02-01,A84423937R,2108.211696,NewSouthWales
1,1978-03-01,A84423937R,2109.021527,NewSouthWales
2,1978-04-01,A84423937R,2109.854462,NewSouthWales
3,1978-05-01,A84423937R,2111.157961,NewSouthWales
4,1978-06-01,A84423937R,2112.979788,NewSouthWales


#### Step3. join part of metadata with dataset contain date and values by series_id

In [135]:
mata_data_updated_columns.head()

NameError: name 'mata_data_updated_columns' is not defined

In [104]:
# analys the values in unit, data_type, frequency, collection_month, no_obs
unit_values = set(mata_data_updated_columns['unit'])
data_type_values = set(mata_data_updated_columns['data_type'])
frequency_values = set(mata_data_updated_columns['frequency'])
collection_month_values = set(mata_data_updated_columns['collection_month'])
no_obs_values = set(mata_data_updated_columns['no_obs'])
print(unit_values, data_type_values, frequency_values, collection_month_values, no_obs_values)

{'000', 'Percent'} {'PERCENT', 'STOCK'} {'Month'} {1} {576}


In [105]:
# select column to merge
selected_metadata = mata_data_updated_columns[['variable', 'gender', 'series_type', 'series_id']]
selected_metadata.head()

,variable,gender,series_type,series_id
0,Employed total,Persons,Trend,A84423937R
1,Employed total,Persons,Seasonally Adjusted,A84423265K
2,Employed total,Persons,Original,A84423601K
3,Employed total,Males,Trend,A84423825W
4,Employed total,Males,Seasonally Adjusted,A84423153T


In [108]:
# Merge df1 and df2 using the shared column 'series_id'
final_dataset_NSW = pd.merge(value_melted, selected_metadata, on='series_id', how='left')
final_dataset_NSW = final_dataset_NSW[['date','series_id', 'state', 'variable', 'gender', 'series_type', 'value']]
final_dataset_NSW.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84423937R,NewSouthWales,Employed total,Persons,Trend,2108.211696
1,1978-03-01,A84423937R,NewSouthWales,Employed total,Persons,Trend,2109.021527
2,1978-04-01,A84423937R,NewSouthWales,Employed total,Persons,Trend,2109.854462
3,1978-05-01,A84423937R,NewSouthWales,Employed total,Persons,Trend,2111.157961
4,1978-06-01,A84423937R,NewSouthWales,Employed total,Persons,Trend,2112.979788


In [107]:
final_dataset_NSW.shape

(48384, 6)

In [293]:
f = round(final_dataset_NSW['value'].astype(float).sum(), 5)
e = round(EmploymentByState_NewSouthWales_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 41072624.73291, original value sum:41072624.73291, there is no value missing: True


### B. EmploymentByState_NorthernTerritory

#### Observe and Analyse the dataset

In [43]:
dataframes['EmploymentByState_NorthernTerritory_Original'].head()

,Unnamed: 0,Employed total ; Persons ;,Employed total ; > Males ;,Employed total ; > Females ;,> Employed full-time ; Persons ;,> Employed full-time ; > Males ;,> Employed full-time ; > Females ;,Employment to population ratio ; Persons ;,Employment to population ratio ; > Males ;,Employment to population ratio ; > Females ;,...,Unemployed total ; > Females ;,Unemployment rate ; Persons ;,Unemployment rate ; > Males ;,Unemployment rate ; > Females ;,Labour force total ; Persons ;,Labour force total ; > Males ;,Labour force total ; > Females ;,Participation rate ; Persons ;,Participation rate ; > Males ;,Participation rate ; > Females ;
0,Unit,000,000,000,000,000,000,Percent,Percent,Percent,...,000,Percent,Percent,Percent,000,000,000,Percent,Percent,Percent
1,Series Type,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,...,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted
2,Data Type,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,PERCENT,PERCENT,PERCENT,...,STOCK,PERCENT,PERCENT,PERCENT,STOCK,STOCK,STOCK,PERCENT,PERCENT,PERCENT
3,Frequency,Month,Month,Month,Month,Month,Month,Month,Month,Month,...,Month,Month,Month,Month,Month,Month,Month,Month,Month,Month
4,Collection Month,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [44]:
dataframes['EmploymentByState_NorthernTerritory_Original'].shape

(585, 22)

In [45]:
dataframes['EmploymentByState_NorthernTerritory_Original'].iloc[:9]

,Unnamed: 0,Employed total ; Persons ;,Employed total ; > Males ;,Employed total ; > Females ;,> Employed full-time ; Persons ;,> Employed full-time ; > Males ;,> Employed full-time ; > Females ;,Employment to population ratio ; Persons ;,Employment to population ratio ; > Males ;,Employment to population ratio ; > Females ;,...,Unemployed total ; > Females ;,Unemployment rate ; Persons ;,Unemployment rate ; > Males ;,Unemployment rate ; > Females ;,Labour force total ; Persons ;,Labour force total ; > Males ;,Labour force total ; > Females ;,Participation rate ; Persons ;,Participation rate ; > Males ;,Participation rate ; > Females ;
0,Unit,000,000,000,000,000,000,Percent,Percent,Percent,...,000,Percent,Percent,Percent,000,000,000,Percent,Percent,Percent
1,Series Type,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,...,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted,Seasonally Adjusted
2,Data Type,STOCK,STOCK,STOCK,STOCK,STOCK,STOCK,PERCENT,PERCENT,PERCENT,...,STOCK,PERCENT,PERCENT,PERCENT,STOCK,STOCK,STOCK,PERCENT,PERCENT,PERCENT
3,Frequency,Month,Month,Month,Month,Month,Month,Month,Month,Month,...,Month,Month,Month,Month,Month,Month,Month,Month,Month,Month
4,Collection Month,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
5,Series Start,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,...,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00,1978-02-01 00:00:00
6,Series End,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,...,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00,2026-01-01 00:00:00
7,No. Obs,576,576,576,576,576,576,576,576,576,...,576,576,576,576,576,576,576,576,576,576
8,Series ID,A84423335F,A84423223L,A84423447X,A84423343F,A84423231L,A84423455X,A84423342C,A84423230K,A84423454W,...,A84423448A,A84423340X,A84423228X,A84423452T,A84423337K,A84423225T,A84423449C,A84423341A,A84423229A,A84423453V


#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Step1. Turn the first to eighth row index into mata data

In [165]:
EmploymentByState_NorthernTerritory = dataframes['EmploymentByState_NorthernTerritory_Original']
NorthTerritory_metadata = extract_metadata(EmploymentByState_NorthernTerritory)

#### Step2. use series ID as column, prepare to melt the rest of dataset including series id, date, and values

In [266]:
# check if series_id has no duplicate
duplicate_rows = NorthTerritory_metadata[NorthTerritory_metadata.duplicated(subset=['series_id'], keep=False)]
duplicate_rows.shape

(0, 11)

In [267]:
state = 'NorthernTerritory'
value_melted = melt_and_update_columns(EmploymentByState_NorthernTerritory_Original, state)
value_melted

,date,series_id,value,state
0,1978-02-01,A84423335F,41.624292,NorthernTerritory
1,1978-03-01,A84423335F,45.341661,NorthernTerritory
2,1978-04-01,A84423335F,44.021738,NorthernTerritory
3,1978-05-01,A84423335F,45.769913,NorthernTerritory
4,1978-06-01,A84423335F,42.605502,NorthernTerritory
...,...,...,...,...
12091,2025-09-01,A84423453V,72.001683,NorthernTerritory
12092,2025-10-01,A84423453V,71.069717,NorthernTerritory
12093,2025-11-01,A84423453V,71.838167,NorthernTerritory
12094,2025-12-01,A84423453V,72.496493,NorthernTerritory


#### Step3. join part of metadata and values by series_id

In [268]:
# select the variables in the metadata
columns = ['variable', 'gender', 'series_type', 'series_id']
order = ['date', 'series_id', 'state', 'variable', 'gender', 'series_type', 'value']

# merge
final_dataset_NT = merge_metadata_and_melted_dataset(
    metadata = NorthTerritory_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_NT = final_dataset_NT[order]

# print
final_dataset_NT.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84423335F,NorthernTerritory,Employed total,Persons,Seasonally Adjusted,41.624292
1,1978-03-01,A84423335F,NorthernTerritory,Employed total,Persons,Seasonally Adjusted,45.341661
2,1978-04-01,A84423335F,NorthernTerritory,Employed total,Persons,Seasonally Adjusted,44.021738
3,1978-05-01,A84423335F,NorthernTerritory,Employed total,Persons,Seasonally Adjusted,45.769913
4,1978-06-01,A84423335F,NorthernTerritory,Employed total,Persons,Seasonally Adjusted,42.605502


In [292]:
f = round(final_dataset_NT['value'].astype(float).sum(), 5)
e = round(EmploymentByState_NorthernTerritory_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 580047.30024, original value sum:580047.30024, there is no value missing: True


### C. EmploymentByState_Queensland

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Code

In [255]:
EmploymentByState_Queensland = dataframes['EmploymentByState_Queensland_Original'].copy()

# Step1 
Queensland_metadata = extract_metadata(EmploymentByState_Queensland)
Queensland_metadata.head()

# Step2
state = 'Queensland'
value_melted = melt_and_update_columns(EmploymentByState_Queensland, state)

# Step3
# select the variables in the metadata
columns = ['variable', 'gender', 'series_type', 'series_id']
order = ['date', 'series_id', 'state', 'variable', 'gender', 'series_type', 'value']

# merge
final_dataset_QL = merge_metadata_and_melted_dataset(
    metadata = Queensland_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_QL = final_dataset_QL[order]

# print
final_dataset_QL.head()


,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84423951K,Queensland,Employed total,Persons,Trend,872.764603
1,1978-03-01,A84423951K,Queensland,Employed total,Persons,Trend,872.336548
2,1978-04-01,A84423951K,Queensland,Employed total,Persons,Trend,872.096895
3,1978-05-01,A84423951K,Queensland,Employed total,Persons,Trend,871.91634
4,1978-06-01,A84423951K,Queensland,Employed total,Persons,Trend,872.08919


In [291]:
f = round(final_dataset_QL['value'].astype(float).sum(), 5)
e = round(EmploymentByState_Queensland_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 23869119.71346, original value sum:23869119.71346, there is no value missing: True


### D. EmploymentByState_SouthAustralia

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Code

In [269]:
EmploymentByState_SouthAustralia= dataframes['EmploymentByState_SouthAustralia_Original'].copy()

# Step1 
SouthAustralia_metadata = extract_metadata(EmploymentByState_SouthAustralia)

# Step2
state = 'SouthAustralia'
value_melted = melt_and_update_columns(EmploymentByState_SouthAustralia, state)

# Step3
# merge
final_dataset_SA = merge_metadata_and_melted_dataset(
    metadata = SouthAustralia_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_SA = final_dataset_SA[order]

# print
final_dataset_SA.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84424035W,SouthAustralia,Employed total,Persons,Trend,555.233447
1,1978-03-01,A84424035W,SouthAustralia,Employed total,Persons,Trend,556.805668
2,1978-04-01,A84424035W,SouthAustralia,Employed total,Persons,Trend,558.167085
3,1978-05-01,A84424035W,SouthAustralia,Employed total,Persons,Trend,559.371365
4,1978-06-01,A84424035W,SouthAustralia,Employed total,Persons,Trend,560.089981


In [290]:
f = round(final_dataset_SA['value'].astype(float).sum(), 5)
e = round(EmploymentByState_SouthAustralia_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 10045634.07174, original value sum:10045634.07174, there is no value missing: True


### E. EmploymentByState_Tasmania

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Code

In [270]:
EmploymentByState_Tasmania= dataframes['EmploymentByState_Tasmania_Original'].copy()

# Step1 
Tasmania_metadata = extract_metadata(EmploymentByState_Tasmania)

# Step2
state = 'Tasmania'
value_melted = melt_and_update_columns(EmploymentByState_Tasmania, state)

# Step3
# merge
final_dataset_T = merge_metadata_and_melted_dataset(
    metadata = Tasmania_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_T = final_dataset_T[order]

# print
final_dataset_T.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84423965X,Tasmania,Employed total,Persons,Trend,165.523304
1,1978-03-01,A84423965X,Tasmania,Employed total,Persons,Trend,165.445856
2,1978-04-01,A84423965X,Tasmania,Employed total,Persons,Trend,165.393696
3,1978-05-01,A84423965X,Tasmania,Employed total,Persons,Trend,165.384243
4,1978-06-01,A84423965X,Tasmania,Employed total,Persons,Trend,165.46602


In [287]:
f = round(final_dataset_T['value'].astype(float).sum(), 5)
e = round(EmploymentByState_Tasmania_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 3514402.03715, original value sum:3514402.03715, there is no value missing: True


### F. EmploymentByState_Victoria

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Code:

In [278]:
EmploymentByState_Victoria = dataframes['EmploymentByState_Victoria_Original'].copy()

# Step1 
Victoria_metadata = extract_metadata(EmploymentByState_Victoria)

# Step2
state = 'Victoria'
value_melted = melt_and_update_columns(EmploymentByState_Victoria, state)

# Step3
# merge
final_dataset_V = merge_metadata_and_melted_dataset(
    metadata = Victoria_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_V = final_dataset_V[order]

# print
final_dataset_V.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84424021J,Victoria,Employed total,Persons,Trend,1638.123654
1,1978-03-01,A84424021J,Victoria,Employed total,Persons,Trend,1642.785934
2,1978-04-01,A84424021J,Victoria,Employed total,Persons,Trend,1646.610122
3,1978-05-01,A84424021J,Victoria,Employed total,Persons,Trend,1649.552186
4,1978-06-01,A84424021J,Victoria,Employed total,Persons,Trend,1650.88087


In [286]:
f = round(final_dataset_V['value'].astype(float).sum(), 5)
e = round(EmploymentByState_Victoria_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 32230680.76857, original value sum:32230680.76857, there is no value missing: True


### G. EmploymentByState_WesternAuds

#### Data processing plan
Note: the EmploymentByState datasets are in same datastructure, so we will use same data process plan via all EmploymentByState datasets with difference states.\
Step1. Turn the first to eighth row index into columns.\
Step2. Use series ID as column, prepare to melt the rest of dataset including series id, date, and values\
Step3. join part of metadata with dataset contain date and values by series_id

#### Code:

In [280]:
EmploymentByState_WesternAuds = dataframes['EmploymentByState_WesternAuds_Original'].copy()

# Step1 
WesternAuds_metadata = extract_metadata(EmploymentByState_WesternAuds)

# Step2
state = 'WesternAuds'
value_melted = melt_and_update_columns(EmploymentByState_WesternAuds, state)

# Step3
# merge
final_dataset_WA = merge_metadata_and_melted_dataset(
    metadata = WesternAuds_metadata,
    melted_dataset = value_melted,
    selected_columns = columns
)

# reorder columns
final_dataset_WA = final_dataset_WA[order]

# print
final_dataset_WA.head()

,date,series_id,state,variable,gender,series_type,value
0,1978-02-01,A84423993J,WesternAuds,Employed total,Persons,Trend,529.789199
1,1978-03-01,A84423993J,WesternAuds,Employed total,Persons,Trend,529.527348
2,1978-04-01,A84423993J,WesternAuds,Employed total,Persons,Trend,529.099262
3,1978-05-01,A84423993J,WesternAuds,Employed total,Persons,Trend,528.615678
4,1978-06-01,A84423993J,WesternAuds,Employed total,Persons,Trend,527.969603


In [282]:
f = round(final_dataset_WA['value'].astype(float).sum(), 5)
e = round(EmploymentByState_WesternAuds_Original.iloc[9:,1:].astype(float).sum().sum(), 5)
print(f'final value sum: {f}, original value sum:{e}, there is no value missing: {f == e}')

final value sum: 13381645.74397, original value sum:13381645.74397, there is no value missing: True


### ERP_Quarterly

#### Observe and Analyse the dataset

In [20]:
ERP_Quarterly_Original = dataframes['ERP_Quarterly_Original'].copy()

In [9]:
for column_name in ERP_Quarterly_Original.columns:
    print(column_name)

STRUCTURE
STRUCTURE_ID
STRUCTURE_NAME
ACTION
MEASURE
Measure
SEX
Sex
AGE
Age
REGION
Region
FREQ
Frequency
TIME_PERIOD
Time Period
OBS_VALUE
Observation Value
UNIT_MEASURE
Unit of Measure
OBS_STATUS
Observation Status
OBS_COMMENT
Observation Comment


In [10]:
A1 = 'OBS_COMMENT'
B1 = 'Observation Comment'
A2 = ERP_Quarterly_Original[A1].unique()
B2 = ERP_Quarterly_Original[B1].unique()
print(f'{A1}:{A2}, {B1}:{B2}')

OBS_COMMENT:[nan], Observation Comment:[nan]


In [11]:

# each number in MEASURE represent a value in measure
A = list(
    ERP_Quarterly_Original[
        ERP_Quarterly_Original['OBS_STATUS'] == 'u'
    ]['Observation Status'].unique()
)
B = list(
    ERP_Quarterly_Original[
        ERP_Quarterly_Original['OBS_STATUS'] == 'q'
    ]['Observation Status'].unique()
)
print(f' u:{A}, q:{B}')

 u:['not applicable'], q:['not available']


In [12]:
#Note!! alot of nulls need to be removed
n = ERP_Quarterly_Original[
    (ERP_Quarterly_Original['OBS_STATUS'] != 'p') 
    & (ERP_Quarterly_Original['OBS_STATUS'] != 'u')
]['OBS_VALUE'].shape[0]
print(f'the number of observation that is not null: {n}')

the number of observation that is not null: 1636200


In [13]:
# each number in MEASURE represent a value in measure
A = list(
    ERP_Quarterly_Original[
        ERP_Quarterly_Original['MEASURE'] == 1
    ]['Measure'].unique()
)
B = list(
    ERP_Quarterly_Original[
        ERP_Quarterly_Original['MEASURE'] == 2
    ]['Measure'].unique()
)
C = list(
    ERP_Quarterly_Original[
        ERP_Quarterly_Original['MEASURE'] == 3
    ]['Measure'].unique()
)
print(f'MEASURE with corresponding Measure - 1:{A}, 2:{B}, 3:{C}')

MEASURE with corresponding Measure - 1:['Estimated Resident Population'], 2:['ERP Change Over Previous Year'], 3:['ERP Percentage Change Over Previous Year']


In [14]:
ERP_Quarterly_Original.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,MEASURE,Measure,SEX,Sex,AGE,Age,...,TIME_PERIOD,Time Period,OBS_VALUE,Observation Value,UNIT_MEASURE,Unit of Measure,OBS_STATUS,Observation Status,OBS_COMMENT,Observation Comment
0,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,3,ERP Percentage Change Over Previous Year,2,Females,24,24,...,1981-Q3,NaN,NaN,NaN,PCT,Percent,u,not applicable,NaN,NaN
1,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,3,ERP Percentage Change Over Previous Year,2,Females,24,24,...,1981-Q4,NaN,NaN,NaN,PCT,Percent,u,not applicable,NaN,NaN
2,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,3,ERP Percentage Change Over Previous Year,2,Females,24,24,...,1982-Q1,NaN,NaN,NaN,PCT,Percent,u,not applicable,NaN,NaN
3,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,3,ERP Percentage Change Over Previous Year,2,Females,24,24,...,1982-Q2,NaN,NaN,NaN,PCT,Percent,u,not applicable,NaN,NaN
4,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,3,ERP Percentage Change Over Previous Year,2,Females,24,24,...,1982-Q3,NaN,3.02,NaN,PCT,Percent,NaN,NaN,NaN,NaN


Note: columns to keep\
Keep the following columns in Metadata: STRUCTURE, STRUCTURE_ID, STRUCTURE_NAME, ACTION, Measure, Frequency, Unit of Measure\
Keep the following columns in dataset: Measure, Sex, Age, Region, TIME_PERIOD, OBS_VALUE

#### Data processing plan
Step1. extract columns to get metadata. \
Step2. extract columns to get main dataset.

#### Code:

##### Step1. Extract columns to get metadata.

In [15]:
ERP_Quarterly_metadata = ERP_Quarterly_Original[['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'Measure', 'Frequency', 'Unit of Measure']].drop_duplicates()

##### Step2. Extact columns to get main data.

In [16]:
ERP_Quarterly = ERP_Quarterly_Original[['Measure', 'Sex', 'Age', 'Region', 'TIME_PERIOD', 'OBS_VALUE']].drop_duplicates()

##### Step3. reset columns

In [17]:
update_column_names(ERP_Quarterly_metadata)

,structure,structure_id,structure_name,action,measure,frequency,unit_of_measure
0,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,ERP Percentage Change Over Previous Year,Quarterly,Percent
342,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,Estimated Resident Population,Quarterly,Persons
855,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,ERP Change Over Previous Year,Quarterly,Persons


In [18]:
ERP_Quarterly_metadata.head()

,structure,structure_id,structure_name,action,measure,frequency,unit_of_measure
0,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,ERP Percentage Change Over Previous Year,Quarterly,Percent
342,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,Estimated Resident Population,Quarterly,Persons
855,DATAFLOW,ABS:ERP_Q(1.0.0),"Quarterly Population Estimates (ERP), by State...",I,ERP Change Over Previous Year,Quarterly,Persons


In [19]:
update_column_names(ERP_Quarterly)
ERP_Quarterly.head()

,measure,sex,age,region,time_period,obs_value
0,ERP Percentage Change Over Previous Year,Females,24,Western Australia,1981-Q3,NaN
1,ERP Percentage Change Over Previous Year,Females,24,Western Australia,1981-Q4,NaN
2,ERP Percentage Change Over Previous Year,Females,24,Western Australia,1982-Q1,NaN
3,ERP Percentage Change Over Previous Year,Females,24,Western Australia,1982-Q2,NaN
4,ERP Percentage Change Over Previous Year,Females,24,Western Australia,1982-Q3,3.02


### Job_vacancies

In [65]:
Job_vacancies_Original = dataframes['Job_vacancies_Original'].copy()

In [66]:
Job_vacancies_Original.head()

,"Job vacancies, states and territories, original",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NSW ('000),Vic ('000),Qld ('000),SA ('000),WA ('000),Tas ('000),NT ('000),ACT ('000)
1,Nov-20,82.7,58.4,47.9,15.6,37.4,5.1,4.2,7.9
2,Feb-21,94,65.6,53.5,19.3,40,5.7,4.8,8.2
3,May-21,112.8,87.6,67.2,21.3,46.2,6.5,5,8.8
4,Aug-21,95.2,81.3,70.6,18,52.1,5.8,6.1,8


1.Columns are not neccessary, they can be removed.\
2.The first row is Australian states.\
3.The first column is data of the observation. 

#### Data processing plan
Step1. replce the current column name with first rows, and name first column 'date'.\
Step2. melt the dataset and make the current column name a column named state.\
Step3. extract month from the date, remove('000) in state, and times values with 1000.

#### Code:

##### Step1. replce the current column name with first rows, and name first column 'date'.

In [125]:
# replace column name
Job_vacancies_rename_column = Job_vacancies_Original.copy()
Job_vacancies_rename_column.columns = Job_vacancies_rename_column.iloc[0]
Job_vacancies_rename_column = Job_vacancies_rename_column.drop([0, 22, 23])

# rename first column
first_column = Job_vacancies_rename_column.columns[0]
Job_vacancies_rename_column = Job_vacancies_rename_column.rename(columns = {first_column:'date'})

##### Step2. melt the dataset and make the current column name a column named state.

In [126]:
Job_vacancies_melt = Job_vacancies_rename_column.copy()
Job_vacancies_melt = Job_vacancies_melt.melt(id_vars='date', var_name='state', value_name='value', ignore_index=True)

##### Step3. extract month from the date, remove('000) in state, and times values with 1000.

In [130]:
Job_vacancies = Job_vacancies_melt.copy()
Job_vacancies['month'] = Job_vacancies['date'].str.split('-').str[0]
Job_vacancies['state'] = Job_vacancies['state'].str.split('(').str[0]
Job_vacancies['value'] = Job_vacancies['value']*1000
Job_vacancies = Job_vacancies[['state', 'month', 'value']]

### Combine datasets

In [ ]:
LabourForce_Current.columns

In [ ]:
EmployPopulationRatial.columns

## Archive Codes

### Labor force status and growth change

#### Observe and Analyse the dataset

In [ ]:
LabourForce_CurrentOriginal.head()

#### Data Reframe plan

1. Remove uneccessary rows and rename the columns\
2. Extract Month and year

#### 1. Remove uneccessary rows and rename the columns

In [ ]:
# make a copy
LabourForce_Current = LabourForce_CurrentOriginal.copy()

In [ ]:
LabourForce_Current_PreColNames = LabourForce_Current.columns
LabourForce_Current_NewColNames = LabourForce_Current.iloc[2].to_numpy()

In [ ]:
LabourForce_Current_RemoveRows = LabourForce_Current.iloc[3:,:7]
LabourForce_Current_RemoveRows.head()

In [ ]:
LabourForce_Current_RenameColumnNames = LabourForce_Current_RemoveRows.rename(columns = {LabourForce_Current_PreColNames[0]:LabourForce_Current_NewColNames[0],
                                                                                         LabourForce_Current_PreColNames[1]:LabourForce_Current_NewColNames[1],
                                                                                         LabourForce_Current_PreColNames[2]:LabourForce_Current_NewColNames[2],
                                                                                         LabourForce_Current_PreColNames[3]:LabourForce_Current_NewColNames[3],
                                                                                         LabourForce_Current_PreColNames[4]:LabourForce_Current_NewColNames[4],
                                                                                         LabourForce_Current_PreColNames[5]:LabourForce_Current_NewColNames[5],
                                                                                         LabourForce_Current_PreColNames[6]:LabourForce_Current_NewColNames[6],
                                                                                        })

In [ ]:
LabourForce_Current_RenameColumnNames = LabourForce_Current_RenameColumnNames.rename(columns = {"Month":"Date", "State and territory (STT): ASGS (2011)":"State", "Labour force status - current month":"status_current_month", "Labour force status - previous month":"status_previous_month", "Persons - current month ('000)":"value_current_month"})

In [ ]:
LabourForce_Current_RenameColumnNames.head()

#### 2. Extract Month and year

In [ ]:
LabourForce_Current_MonthYear = LabourForce_Current_RenameColumnNames.copy()

In [ ]:
LabourForce_Current_MonthYear.dtypes

In [ ]:
LabourForce_Current_MonthYear.head()

In [ ]:
LabourForce_Current_MonthYear['Date'][3]

Observed a mix of object and datetime. The plan is to convert the column to actuall datetime formate

In [ ]:
LabourForce_Current_MonthYear['Date'] = pd.to_datetime(LabourForce_Current_MonthYear['Date'])

In [ ]:
# extract month and year
LabourForce_Current_MonthYear['Year'] = LabourForce_Current_MonthYear['Date'].dt.year
LabourForce_Current_MonthYear['Month'] = LabourForce_Current_MonthYear['Date'].dt.month

In [ ]:
LabourForce_Current_MonthYear.head()

In [ ]:
#Drop Date
LabourForce_Current_DateDroped = LabourForce_Current_MonthYear.drop('Date', axis=1)

In [ ]:
LabourForce_Current = LabourForce_Current_DateDroped.copy()

### Employment to population Ration

#### Observe and Analyse the dataset

In [ ]:
EmployPopulationRatial_Original

In [ ]:
EmployPopulationRatial = EmployPopulationRatial_Original.copy()

Observation:\
The size of the data set is 124X1. The only column is named Employment-to-population ratio. Under the column there are three sub-columns. The first column is datetime, and the rest are the numerical data. The last two rows are not the value of the datasets.

#### Data processing plan

Step1 - reset index and the column names.\
Step2 - remove the last two rows.\
Step3 - extract month and year.\
Step4 - convert the datatypes of month and year to numeric.\
Step5 - convert data in seasonally adjusted from percentage to decimal

#### Step1 - reset index and the column names.

In [ ]:
EmployPopulationRatial_ResetIndex = EmployPopulationRatial.reset_index()

In [ ]:
EmployPopulationRatial_ResetColumnName = EmployPopulationRatial_ResetIndex.copy()
EmployPopulationRatial_ResetColumnName.columns = EmployPopulationRatial_ResetColumnName.iloc[0].tolist()
EmployPopulationRatial_ResetColumnName = EmployPopulationRatial_ResetColumnName.iloc[1:]

In [ ]:
OriginalColumnNames = EmployPopulationRatial_ResetColumnName.columns
EmployPopulationRatial_ResetColumnName = EmployPopulationRatial_ResetColumnName.rename(columns = {OriginalColumnNames[0]:'Date', OriginalColumnNames[1]:'Employment_Population_Ratio', OriginalColumnNames[2]:'Ratio_SeasonallyAdjusted'})

In [ ]:
EmployPopulationRatial_ResetColumnName.head()

#### Step2 - remove the last two rows.

In [ ]:
EmployPopulationRatial_RemovedRows = EmployPopulationRatial_ResetColumnName.iloc[:121] 
EmployPopulationRatial_RemovedRows

#### Step3 - extract month and year.

In [ ]:
EmployPopulationRatial_ExtractMonthYear = EmployPopulationRatial_RemovedRows.copy()

In [ ]:
MonthYear = EmployPopulationRatial_ExtractMonthYear['Date'].str.split('-', expand=True)

In [ ]:
MonthYear.columns

In [ ]:
EmployPopulationRatial_ExtractMonthYear['Month_Name'] = MonthYear[0]
EmployPopulationRatial_ExtractMonthYear['Year'] = '20' + MonthYear[1]

In [ ]:
EmployPopulationRatial_ExtractMonthYear['Month'] = pd.to_datetime(EmployPopulationRatial_ExtractMonthYear['Month_Name'], format='%b').dt.month

In [ ]:
# drop Date, Month_Name
EmployPopulationRatial = EmployPopulationRatial_ExtractMonthYear.drop(columns = ['Date', 'Month_Name'])

In [ ]:
EmployPopulationRatial.head()

#### Step4 - convert the datatypes of month and year to numeric.

In [ ]:
EmployPopulationRatial['Year'] = EmployPopulationRatial['Year'].astype(int)

In [ ]:
EmployPopulationRatial['Month'] = EmployPopulationRatial['Month'].astype(int)

#### Step5 - convert data in seasonally adjusted from percentage to decimal

In [ ]:
#convert string to float then transfrom the data
EmployPopulationRatial['Ratio_SeasonallyAdjusted'] = EmployPopulationRatial['Ratio_SeasonallyAdjusted'].astype(float)/100

In [ ]:
#drop first column
EmployPopulationRatial_cleaned = EmployPopulationRatial[['Ratio_SeasonallyAdjusted', 'Year', 'Month']]

In [ ]:
EmployPopulationRatial_cleaned.head(5)

### CPI_MonthlyPath(Archive)

#### Observe and Analyse the dataset

In [ ]:
CPI_MonthlyPath = CPI_MonthlyOriginal.copy()
CPI_MonthlyPath.head()

Observation:\
Data set type - structural data, wide data sets
Size - 30rows X 133 columns
columns- first column contains the information of each rows without a column name which suppose to be the column names, including but not limited to Unit, series type, data type, etc. From the seconde column to the last column, the column name contains the industry, including all groups, non-alcoholic beverages, bread and cereal product, etc. 
Data types - the data sets contain categorical, numerical and datetime datasets

#### Data processing plan

Step1 - transfer the wide data to long.\
Step2 - extract the industries info and name the column industry.\
Step3 - extract the Month and Year information from the datetime (located in the first row originally), named as Month and Year.\
Step4 - deleted the columns/information that is not neceessary, for example colleciton month, series start, etc.\

#### Step1 - transfer the wide data to long.

In [ ]:
CPT_Wide = CPI_MonthlyPath.T

#### Step2 - extract the industries info and name the column industry.

In [ ]:
CPT_Wide.head()

In [ ]:
CPT_Wide_resetIndex = CPT_Wide.reset_index()

In [ ]:
CPT_Wide_resetIndex.head()

In [ ]:
#make first row as column name
CPT_Wide_resetIndex.columns = CPT_Wide_resetIndex.iloc[0]
CPT_DropFirstRow = CPT_Wide_resetIndex.drop(0, axis = 0)

In [ ]:
#name the first column industry
firstColumnName = CPT_DropFirstRow.columns[0]
rename = 'Industry'
CPT_rename = CPT_DropFirstRow.rename(columns = {firstColumnName:rename})

In [ ]:
#reset index
CPT_Wide_resetIndex = CPT_rename.reset_index(drop=True)

In [ ]:
#extract the industry
CPT_Wide_resetIndex['Industry'] = CPT_Wide_resetIndex['Industry'].str.split(' ; ', expand=True)[1]

#### Step3 - extract the Month and Year information from the datetime (located in the first row originally), named as Month and Year.

In [ ]:
CPT_Wide_resetIndex.columns[10:]

In [ ]:
#Get the columns that are not CPT with dates
CPT_nonDateColumnNames = CPT_Wide_resetIndex.columns[:10].tolist()
CPT_nonDateColumnNames

In [ ]:
#Melt the dataframe
CPT_MeltedDateTime = pd.melt(CPT_Wide_resetIndex, id_vars=CPT_nonDateColumnNames, var_name = 'DateTime', value_name = 'CPT_Changes')

In [ ]:
CPT_MeltedDateTime.head()

In [ ]:
CPT_MeltedDateTime.dtypes

In [ ]:
CPT_MeltedDateTime['Year'] = CPT_MeltedDateTime['DateTime'].dt.year

In [ ]:
CPT_MeltedDateTime['Month'] = CPT_MeltedDateTime['DateTime'].dt.month

In [ ]:
CPT_MeltedDateTime.head()

In [ ]:
CPT_MeltedDateTime.dtypes

#### Step4 - deleted the columns/information that is not neceessary, for example colleciton month, series start, etc.

In [ ]:
CPT = CPT_MeltedDateTime.copy()

In [ ]:
CPT['No. Obs'].unique()

Remove Unit, Series Type, Data Type, Frequency, Collection Month, No.Obs, because there is only one value in these columns

In [ ]:
dropedColumns = ['Unit', 'Series Type', 'Data Type', 'Frequency', 'Collection Month', 'No. Obs']

In [ ]:
CPT_DropedColumns = CPT.drop(columns = dropedColumns, axis = 1) 

Also dropped Series Start, Series end and Series ID cause its not relavant to CPT changes. Dropped DateTime Cause we already have Year and Month

In [ ]:
dropedColumns = ['Series Start', 'Series End', 'DateTime', 'Series ID']

In [ ]:
CPT_DropedDates = CPT_DropedColumns.drop(columns = dropedColumns, axis = 1)

In [ ]:
CPT_DropedDates.shape

In [ ]:
CPT = CPT_DropedDates.copy()